# ParkVision · EfficientNetB0 · vacant / vehicle / obstruction

**Upload this notebook to Google Colab → select T4 GPU → Run all.** It is self-contained; no repository upload is needed. It trains only EfficientNetB0 and exports a candidate compatible with ParkVision's existing saved parking boxes.

| Output | Meaning inside ONE fixed parking-region crop |
|---|---|
| 0 `vacant` | No vehicle or non-vehicle obstruction occupies the intended usable area. |
| 1 `occupied` | A vehicle occupies the position. |
| 2 `blocked` | A non-vehicle obstruction (for example a cone or box) occupies the intended usable area. |

A ceiling/wall/background object outside the usable parking area is **not** a blockage. When an actual obstruction and vehicle both affect a position, label it **blocked**. Public positive crops with mixed vehicle/obstruction annotations are excluded rather than guessing that relationship. Label ambiguous site examples carefully or exclude them before splitting.

Defaults download actual parking examples (PKLot), labeled vehicles/people/boxes and other objects (Open Images), and traffic cones (TraCon). **These provide a public-data baseline, not proof of site performance or arbitrary-object detection.** Add representative fixed-box site crops to teach what counts as blocking your parking area. The system cannot determine physical parkability or vehicle clearance from a classifier alone. No CCTV connection is needed to train or test this.

## 1 · Install
Use **Runtime → Change runtime type → T4 GPU**. Run in a fresh runtime. Data download can take several minutes and several GB, especially PKLot. Authorize Drive to preserve checkpoints. The full GPU run must be performed by you; creating this notebook does not train or replace the installed app model.

In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'tensorflow==2.20.0', 'keras==3.13.2', 'Pillow>=11.3,<13',
    'pandas>=2.2,<3', 'scikit-learn>=1.5,<2', 'matplotlib>=3.9,<4',
    'kagglehub>=0.3,<1', 'requests>=2.32,<3', 'tqdm>=4.66,<5'])

## 2 · Settings
The defaults run without uploading your own data. For stronger site results, add the three-class ZIP described in section 4. `WARM_START_UPLOAD=True` optionally loads only the backbone from your existing EfficientNetB0 `.keras` model (or its export ZIP); the three-class head is newly trained. Keep every image used by that old model out of the new independent test set. Leave False if that earlier data provenance is unclear. ImageNet initialization is the default.

In [ ]:
CONFIG = {
    'SEED': 42, 'REQUIRE_GPU': True, 'USE_GOOGLE_DRIVE': True,
    'USE_PKLOT': True, 'USE_OPEN_IMAGES': True, 'USE_TRACON': True,
    'PKLOT_ROOT': '', 'SITE_ROOT': '', 'UPLOAD_SITE_ZIP': False,
    'WARM_START_UPLOAD': False,
    'VEHICLE_NAMES': ['Car', 'Van', 'Truck', 'Bus', 'Motorcycle', 'Bicycle'],
    'OBSTRUCTION_NAMES': ['Person', 'Man', 'Woman', 'Boy', 'Girl', 'Box', 'Chair', 'Cart',
                          'Waste container', 'Luggage and bags', 'Plastic bag', 'Handbag'],
    'PKLOT_PER_CLASS': {'train': 2500, 'val': 400, 'test': 400},
    'OPENIMAGES_PER_CLASS': {'train': 1000, 'val': 200, 'test': 200},
    'TRACON_CAPS': {'train': 350, 'val': 70, 'test': 70},
    'SOURCE_WEIGHTS': {'pklot': 1.0, 'openimages': 1.5, 'tracon': 0.6, 'site': 4.0},
    'MAX_TRAIN_SAMPLES_PER_EPOCH': 16000, 'BATCH_SIZE': 32,
    'HEAD_EPOCHS': 5, 'FINE_TUNE_EPOCHS': 10,
    'REVIEW_THRESHOLD': 0.80, 'RUN_MANUAL_IMAGE_TEST': False,
}

In [ ]:
import os, json, random
from pathlib import Path
from datetime import datetime, timezone
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from PIL import Image, ImageOps
from IPython.display import display

tf.keras.utils.set_random_seed(CONFIG['SEED'])
gpus = tf.config.list_physical_devices('GPU')
print('TensorFlow:', tf.__version__, 'Keras:', tf.keras.__version__, 'GPU:', gpus)
if CONFIG['REQUIRE_GPU'] and not gpus:
    raise RuntimeError('Choose Runtime → Change runtime type → T4 GPU, then Run all.')
for gpu in gpus:
    try: tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError: pass
if CONFIG['USE_GOOGLE_DRIVE']:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = Path('/content/drive/MyDrive/ParkVision_ThreeClass_EfficientNetB0')
else:
    PROJECT_DIR = Path('/content/ParkVision_ThreeClass_EfficientNetB0')
    print('Drive disabled: runtime files disappear on disconnect; download the export.')
CACHE_DIR = Path('/content/parkvision_threeclass_cache')
CROP_DIR = CACHE_DIR / 'crops'
RUN_DIR = PROJECT_DIR / 'runs' / datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
for directory in (CACHE_DIR, CROP_DIR, RUN_DIR): directory.mkdir(parents=True, exist_ok=True)
(RUN_DIR / 'config.json').write_text(json.dumps(CONFIG, indent=2))
print('Checkpoints/reports:', RUN_DIR)
print('Class order is FIXED: 0 vacant, 1 occupied, 2 blocked.')

## 3 · Prepare public data
Run the helper cells unchanged. All helpers are embedded below. Source images/scenes are assigned to splits **before** cropping. Exact conflicting labels and cross-split duplicates are removed later. Open Images backgrounds exclude all annotated objects, but missing annotations remain possible; inspect the labels. TraCon supplies cone positives only, since its labels cannot establish that a background is clear of other objects.

In [ ]:
"""Training-data utilities embedded into the standalone ParkVision Colab notebook."""
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from functools import lru_cache
from pathlib import Path
import hashlib
import json
import math
import random
import re
import shutil
import zipfile

import numpy as np
import pandas as pd
from PIL import Image, ImageOps
import requests
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from tqdm.auto import tqdm

SPLITS = ('train', 'val', 'test')
CLASS_NAMES = ['unoccupied', 'occupied']  # Explicit numeric order; never inferred from folders.
IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png', '.webp'}


def clean_bbox(box, image_width, image_height, min_size=8):
    if not isinstance(box, (list, tuple)) or len(box) != 4:
        return None
    if any(isinstance(x, bool) or not isinstance(x, (int, float)) or not math.isfinite(x) for x in box):
        return None
    x, y, width, height = box
    if width <= 0 or height <= 0:
        return None
    left, top = max(0, math.floor(x)), max(0, math.floor(y))
    right, bottom = min(image_width, math.ceil(x + width)), min(image_height, math.ceil(y + height))
    return (left, top, right, bottom) if right - left >= min_size and bottom - top >= min_size else None


def overlaps(a, b):
    return min(a[2], b[2]) > max(a[0], b[0]) and min(a[3], b[3]) > max(a[1], b[1])


def negative_crop(width, height, vehicle_boxes, rng, attempts=60):
    """Never label a proposed negative if it overlaps an annotated target vehicle."""
    for _ in range(attempts):
        cw = max(16, round(width * rng.uniform(0.22, 0.75)))
        ch = max(16, round(height * rng.uniform(0.22, 0.75)))
        if cw > width or ch > height:
            continue
        x, y = rng.randint(0, width - cw), rng.randint(0, height - ch)
        box = (x, y, x + cw, y + ch)
        if not any(overlaps(box, vehicle) for vehicle in vehicle_boxes):
            return box
    return None


def frame_group(filename):
    """Keep PKLot capture days together when dates are recoverable from names."""
    name = Path(filename).name
    date = re.search(r'(20\d{2})[-_](\d{2})[-_](\d{2})', name)
    if date:
        return 'pklot_day_' + '-'.join(date.groups()), 'capture_day'
    stem = re.sub(r'\.rf\.[^.]+', '', Path(name).stem)
    return 'pklot_frame_' + stem, 'source_frame_only'


def grouped_split(frame, group_column='group_id', seed=42):
    """Assign complete groups before generating crops, with train/val/test isolation."""
    frame = frame.reset_index(drop=True).copy()
    if frame[group_column].nunique() < 3:
        raise ValueError('Need at least three independent capture-day/recording groups for train, val and test.')
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=seed)
    train_idx, holdout_idx = next(splitter.split(frame, groups=frame[group_column]))
    holdout = frame.iloc[holdout_idx]
    if holdout[group_column].nunique() < 2:
        groups = sorted(frame[group_column].unique())
        random.Random(seed).shuffle(groups)
        mapping = {group: 'train' for group in groups}
        mapping[groups[-1]], mapping[groups[-2]] = 'test', 'val'
        frame['split'] = frame[group_column].map(mapping)
    else:
        second = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=seed + 1)
        val_idx, test_idx = next(second.split(holdout, groups=holdout[group_column]))
        frame['split'] = 'train'
        frame.loc[holdout.iloc[val_idx].index, 'split'] = 'val'
        frame.loc[holdout.iloc[test_idx].index, 'split'] = 'test'
    assert frame.groupby(group_column)['split'].nunique().max() == 1
    return frame


@lru_cache(maxsize=20000)
def file_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def save_crop(image, box, directory, *, source, group_id, split, label, source_id, source_sha):
    crop = image.crop(box).convert('RGB')
    pixel_sha = hashlib.sha256(str(crop.size).encode() + crop.tobytes()).hexdigest()
    directory = Path(directory) / source / split / CLASS_NAMES[label]
    directory.mkdir(parents=True, exist_ok=True)
    target = directory / f'{pixel_sha}.jpg'
    if not target.exists():
        crop.save(target, quality=95, subsampling=0)
    return dict(path=str(target), label=int(label), source=source, group_id=group_id,
                split=split, source_id=str(source_id), source_sha=source_sha, crop_sha=pixel_sha)


def download_public(url, target, max_bytes=400_000_000):
    target = Path(target)
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists() and target.stat().st_size > 0:
        return target
    for attempt in range(3):
        partial = target.with_suffix(target.suffix + '.part')
        try:
            with requests.get(url, stream=True, timeout=(20, 90)) as response:
                response.raise_for_status()
                size = 0
                with partial.open('wb') as output:
                    for chunk in response.iter_content(1024 * 1024):
                        if chunk:
                            size += len(chunk)
                            if size > max_bytes:
                                raise ValueError(f'Download exceeds the configured size limit: {url}')
                            output.write(chunk)
                if not size:
                    raise ValueError('Empty download')
            partial.replace(target)
            return target
        except (requests.RequestException, OSError):
            partial.unlink(missing_ok=True)
            if attempt == 2:
                raise


def safe_extract_zip(archive, target, max_bytes=5_000_000_000):
    target = Path(target).resolve()
    target.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as bundle:
        if sum(item.file_size for item in bundle.infolist()) > max_bytes:
            raise ValueError('Archive expands beyond the configured size limit.')
        for item in bundle.infolist():
            destination = (target / item.filename).resolve()
            if not destination.is_relative_to(target):
                raise ValueError('Archive contains an unsafe path.')
        bundle.extractall(target)
    return target


def build_pklot(root, crop_dir, per_class, seed=42):
    root = Path(root)
    by_basename = defaultdict(list)
    for path in root.rglob('*'):
        if path.suffix.lower() in IMAGE_SUFFIXES:
            by_basename[path.name].append(path)
    records = []
    ignored = Counter()
    for annotation_file in sorted(root.rglob('*.json')):
        if 'annotation' not in annotation_file.name.lower():
            continue
        data = json.loads(annotation_file.read_text())
        if not all(key in data for key in ('images', 'annotations', 'categories')):
            continue
        names = {item['id']: str(item['name']).lower().strip().replace('_', '-').replace(' ', '-') for item in data['categories']}
        categories = {}
        for category_id, name in names.items():
            if name in {'space-empty', 'empty', 'vacant', 'unoccupied', 'free'}:
                categories[category_id] = 0
            elif name in {'space-occupied', 'occupied', 'busy'}:
                categories[category_id] = 1
        if set(categories.values()) != {0, 1}:
            raise ValueError(f'Unrecognized PKLot category names: {names}. Check the labels explicitly.')
        annotations = defaultdict(list)
        for annotation in data['annotations']:
            if annotation.get('category_id') in categories:
                annotations[annotation['image_id']].append(annotation)
        rng = random.Random(seed)
        for image in data['images']:
            relative = image['file_name']
            path = (annotation_file.parent / relative).resolve()
            if not path.is_relative_to(root.resolve()) or not path.is_file():
                alternatives = by_basename[Path(relative).name]
                if len(alternatives) != 1:
                    ignored['missing_or_ambiguous_image'] += 1
                    continue
                path = alternatives[0]
            candidates = annotations[image['id']][:]
            occupied_boxes = [clean_bbox(item.get('bbox'), image['width'], image['height'], min_size=1)
                              for item in candidates if categories[item['category_id']] == 1]
            occupied_boxes = [box for box in occupied_boxes if box is not None]
            rng.shuffle(candidates)
            counts = Counter()
            group, grouping_rule = frame_group(relative)
            for annotation in candidates:
                label = categories[annotation['category_id']]
                if counts[label] >= 2:  # More source frames, fewer correlated crops per frame.
                    continue
                box = clean_bbox(annotation.get('bbox'), image['width'], image['height'])
                if box is None:
                    ignored['invalid_box'] += 1
                    continue
                # PKLot labels parking positions, rather than every visible car.
                # Drop an empty position whose rectangular crop also intersects
                # an occupied position: that is ambiguous for vehicle presence.
                if label == 0 and any(overlaps(box, occupied) for occupied in occupied_boxes):
                    ignored['empty_crop_overlaps_occupied_position'] += 1
                    continue
                counts[label] += 1
                records.append(dict(source_path=str(path), box=box, label=label, group_id=group,
                                    grouping_rule=grouping_rule, source_id=path.name))
    if not records:
        raise ValueError('No usable PKLot COCO annotations found. Set PKLOT_ROOT to the dataset directory.')
    frame = grouped_split(pd.DataFrame(records), seed=seed)
    output, kept_counts = [], Counter()
    for split in SPLITS:
        for label in (0, 1):
            candidates = frame[(frame.split == split) & (frame.label == label)].sample(frac=1, random_state=seed)
            for row in tqdm(candidates.to_dict('records'), desc=f'PKLot {split} {CLASS_NAMES[label]}', leave=False):
                if kept_counts[split, label] >= per_class[split]:
                    break
                try:
                    with Image.open(row['source_path']) as image:
                        image = image.convert('RGB')
                        if row['box'][2] > image.width or row['box'][3] > image.height:
                            ignored['image_annotation_size_mismatch'] += 1
                            continue
                        item = save_crop(image, row['box'], crop_dir, source='pklot', group_id=row['group_id'],
                                         split=split, label=label, source_id=row['source_id'], source_sha=file_hash(row['source_path']))
                        item['grouping_rule'] = row['grouping_rule']
                        output.append(item)
                        kept_counts[split, label] += 1
                except (OSError, ValueError):
                    ignored['unreadable_image'] += 1
    print('PKLot ignored records:', dict(ignored))
    return pd.DataFrame(output)


def build_coco(cache_dir, crop_dir, per_class, vehicle_categories=('car', 'truck', 'bus'), seed=42):
    cache_dir = Path(cache_dir) / 'coco'
    # Official COCO S3 bucket alias uses a valid HTTPS certificate.
    base_url = 'https://s3.amazonaws.com/images.cocodataset.org'
    archive = download_public(base_url + '/annotations/annotations_trainval2017.zip', cache_dir / 'annotations.zip')
    annotation_path = cache_dir / 'instances_val2017.json'
    if not annotation_path.exists():
        with zipfile.ZipFile(archive) as bundle:
            with bundle.open('annotations/instances_val2017.json') as source, annotation_path.open('wb') as target:
                shutil.copyfileobj(source, target)
    data = json.loads(annotation_path.read_text())
    target_ids = {item['id'] for item in data['categories'] if item['name'] in vehicle_categories}
    if len(target_ids) != len(vehicle_categories):
        raise ValueError('Unknown COCO vehicle category. Check VEHICLE_CATEGORIES.')
    annotations = defaultdict(list)
    for annotation in data['annotations']:
        if annotation['category_id'] in target_ids:
            annotations[annotation['image_id']].append(annotation)
    images = pd.DataFrame(data['images'])
    images['has_vehicle'] = images.id.map(lambda image_id: bool(annotations[image_id])).astype(int)
    train, holdout = train_test_split(images, test_size=0.30, random_state=seed, stratify=images.has_vehicle)
    val, test = train_test_split(holdout, test_size=0.5, random_state=seed + 1, stratify=holdout.has_vehicle)
    selected = []
    for split, frame in [('train', train), ('val', val), ('test', test)]:
        for label in (0, 1):
            pool = frame[frame.has_vehicle == label].sample(frac=1, random_state=seed)
            # Reserve some extra images in case a positive annotation is too small/corrupt.
            pool = pool.head(min(len(pool), math.ceil(per_class[split] * 1.3)))
            selected.extend({**item, 'split': split} for item in pool.to_dict('records'))
    def fetch(record):
        name = Path(record['file_name']).name
        target = cache_dir / 'images' / name
        try:
            download_public(f'{base_url}/val2017/{name}', target, max_bytes=30_000_000)
            return record, target
        except (requests.RequestException, OSError, ValueError):
            return record, None
    paths = {}
    with ThreadPoolExecutor(max_workers=8) as executor:
        jobs = [executor.submit(fetch, record) for record in selected]
        for job in tqdm(as_completed(jobs), total=len(jobs), desc='COCO image downloads'):
            record, path = job.result()
            paths[record['id']] = path
    output, counts = [], Counter()
    rng = random.Random(seed)
    # Vehicle scenes first so their verified background patches become hard negatives.
    selected.sort(key=lambda item: (-item['has_vehicle'], item['id']))
    for record in tqdm(selected, desc='COCO vehicle / no-vehicle crops'):
        path, split = paths[record['id']], record['split']
        if path is None:
            continue
        try:
            with Image.open(path) as original:
                image = original.convert('RGB')
                boxes = [clean_bbox(item['bbox'], image.width, image.height, min_size=1) for item in annotations[record['id']]]
                boxes = [box for box in boxes if box is not None]
                candidates = []
                if record['has_vehicle'] and boxes:
                    # Prefer a clearly visible vehicle, retain varied surrounding context.
                    box = max(boxes, key=lambda b: (b[2] - b[0]) * (b[3] - b[1]))
                    if (box[2] - box[0]) * (box[3] - box[1]) >= 400:
                        padding = rng.uniform(0.12, 0.65)
                        bw, bh = box[2] - box[0], box[3] - box[1]
                        expanded = (max(0, math.floor(box[0] - bw * padding)), max(0, math.floor(box[1] - bh * padding)),
                                    min(image.width, math.ceil(box[2] + bw * padding)), min(image.height, math.ceil(box[3] + bh * padding)))
                        candidates.append((1, expanded))
                    background = negative_crop(image.width, image.height, boxes, rng)
                    # Keep room for vehicle-free scenes too; do not fill every
                    # negative slot from vehicle-scene backgrounds alone.
                    if background and counts[split, 0] < math.ceil(per_class[split] / 2):
                        candidates.append((0, background))
                elif not record['has_vehicle']:
                    box = (0, 0, image.width, image.height) if rng.random() < 0.5 else negative_crop(image.width, image.height, [], rng)
                    if box:
                        candidates.append((0, box))
                for label, box in candidates:
                    if counts[split, label] >= per_class[split]:
                        continue
                    output.append(save_crop(image, box, crop_dir, source='coco', group_id=f"coco_val2017_{record['id']}",
                                            split=split, label=label, source_id=record['file_name'], source_sha=file_hash(str(path))))
                    counts[split, label] += 1
        except (OSError, ValueError):
            continue
    if not output:
        raise ValueError('COCO download/crop generation produced no data. Check the download error and retry.')
    return pd.DataFrame(output)


def build_local_crops(root, crop_dir, seed=42):
    """Input: site/day_or_recording_id/{occupied,unoccupied}/*.jpg."""
    root = Path(root)
    rows = []
    labels = {'unoccupied': 0, 'vacant': 0, 'empty': 0, 'occupied': 1}
    for path in root.rglob('*'):
        if path.suffix.lower() not in IMAGE_SUFFIXES or path.parent.name.lower() not in labels:
            continue
        group_folder = path.parent.parent
        if group_folder.resolve() == root.resolve():
            raise ValueError('Local data needs recording/day folders above the class folders. Do not randomly split neighboring video frames.')
        rows.append(dict(source_path=str(path), label=labels[path.parent.name.lower()], group_id='site_' + str(group_folder.relative_to(root))))
    if not rows:
        raise ValueError('No local crops found. Use day01/occupied and day01/unoccupied folders inside the ZIP.')
    frame = grouped_split(pd.DataFrame(rows), seed=seed)
    output = []
    for row in frame.to_dict('records'):
        with Image.open(row['source_path']) as source:
            image = ImageOps.exif_transpose(source).convert('RGB')
            if min(image.size) < 16:
                raise ValueError(f"Local crop is too small: {row['source_path']}")
            output.append(save_crop(image, (0, 0, image.width, image.height), crop_dir, source='site', group_id=row['group_id'],
                                    split=row['split'], label=row['label'], source_id=Path(row['source_path']).name, source_sha=file_hash(row['source_path'])))
    return pd.DataFrame(output)


def audit_manifest(frame):
    frame = frame.copy().reset_index(drop=True)
    if not set(frame.label.unique()) <= {0, 1} or set(frame.split.unique()) != set(SPLITS):
        raise ValueError('Need explicit binary labels and train/val/test splits.')
    collisions = frame.groupby('group_id')['split'].nunique()
    if (collisions > 1).any():
        raise ValueError('A recording/source group appears in more than one split.')
    conflicts = set(frame.groupby('crop_sha')['label'].nunique().loc[lambda values: values > 1].index)
    removed = frame[frame.crop_sha.isin(conflicts)].assign(reason='conflicting_labels')
    frame = frame[~frame.crop_sha.isin(conflicts)].copy()
    priority = {'test': 0, 'val': 1, 'train': 2}
    frame['_priority'] = frame.split.map(priority)
    for column in ('source_sha', 'crop_sha'):
        keep_split = frame.sort_values('_priority').drop_duplicates(column).set_index(column)['split']
        mask = frame.split != frame[column].map(keep_split)
        removed = pd.concat([removed, frame[mask].assign(reason=f'{column}_cross_split')], ignore_index=True)
        frame = frame[~mask].copy()
    frame = frame.drop_duplicates('crop_sha').drop(columns='_priority').reset_index(drop=True)
    for split in SPLITS:
        counts = frame[frame.split == split].label.value_counts()
        if any(counts.get(label, 0) < 5 for label in (0, 1)):
            raise ValueError(f'{split} needs at least five distinct crops per class after leakage cleaning; collect more groups.')
    for source in frame.source.unique():
        for split in SPLITS:
            counts = frame[(frame.source == source) & (frame.split == split)].label.value_counts()
            if any(counts.get(label, 0) < 5 for label in (0, 1)):
                raise ValueError(f'{source}/{split} needs at least five unique crops of each class. Add recording groups containing both classes.')
    assert frame.groupby('source_sha')['split'].nunique().max() == 1
    assert frame.groupby('crop_sha')['split'].nunique().max() == 1
    return frame, removed.drop(columns='_priority', errors='ignore')

In [ ]:
"""Three-class crop preparation; embedded unchanged (apart from imports) in Colab."""
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import hashlib
import math
import random

import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from tqdm.auto import tqdm


STATUS_NAMES = ['vacant', 'occupied', 'blocked']
TRACON_COMMIT = '7ac4af371e826b5124a9b4defaa3327bc817817e'
OI_CLASSES = 'https://storage.googleapis.com/openimages/v7/oidv7-class-descriptions-boxable.csv'
OI_BOXES = 'https://storage.googleapis.com/openimages/v5/validation-annotations-bbox.csv'
OI_CREDITS = 'https://storage.googleapis.com/openimages/2018_04/validation/validation-images-with-rotation.csv'


def save_status_crop(image, box, crop_dir, *, label, source, group_id, split, source_id, source_sha, subtype):
    crop = image.crop(box).convert('RGB')
    digest = hashlib.sha256(str(crop.size).encode() + crop.tobytes()).hexdigest()
    folder = Path(crop_dir) / source / split / STATUS_NAMES[label]
    folder.mkdir(parents=True, exist_ok=True)
    target = folder / f'{digest}.jpg'
    if not target.exists():
        crop.save(target, quality=95, subsampling=0)
    return dict(path=str(target), label=label, source=source, group_id=group_id, split=split,
                source_id=str(source_id), source_sha=source_sha, crop_sha=digest, subtype=subtype)


def padded_box(box, width, height, rng, maximum=0.35):
    padding = rng.uniform(0.08, maximum)
    bw, bh = box[2] - box[0], box[3] - box[1]
    return (max(0, math.floor(box[0] - bw * padding)), max(0, math.floor(box[1] - bh * padding)),
            min(width, math.ceil(box[2] + bw * padding)), min(height, math.ceil(box[3] + bh * padding)))


def build_openimages(cache_dir, crop_dir, caps, vehicle_names, obstruction_names, seed=42):
    """Repartition the official validation source pool by image BEFORE cropping."""
    cache = Path(cache_dir) / 'openimages'
    classes = pd.read_csv(download_public(OI_CLASSES, cache / 'classes.csv'), header=None, names=['id', 'name'])
    lookup = dict(zip(classes.id, classes.name))
    names = set(classes.name)
    missing = (set(vehicle_names) | set(obstruction_names)) - names
    if missing:
        raise ValueError(f'Open Images category names do not exist: {sorted(missing)}. Check the official class list.')
    if set(vehicle_names) & set(obstruction_names):
        raise ValueError('Vehicle and obstruction categories must not overlap.')
    annotations = pd.read_csv(download_public(OI_BOXES, cache / 'boxes.csv'))
    annotations['name'] = annotations.LabelName.map(lookup)
    annotations['label'] = annotations.name.map(lambda name: 1 if name in vehicle_names else 2 if name in obstruction_names else -1)
    images = pd.DataFrame({'source_id': sorted(annotations.ImageID.unique())})
    images['group_id'] = 'oi_image_' + images.source_id
    images = grouped_split(images, seed=seed)
    split_map = dict(zip(images.source_id, images.split))
    positive = annotations[(annotations.label >= 1) & (annotations.IsGroupOf == 0) & (annotations.IsDepiction == 0)].copy()
    positive['split'] = positive.ImageID.map(split_map)
    selected = set()
    # Sample obstruction subtypes separately so boxes do not disappear behind people.
    for split in ('train', 'val', 'test'):
        for label, requested in [(1, vehicle_names), (2, obstruction_names)]:
            for name in requested:
                pool = positive[(positive.split == split) & (positive.name == name)].ImageID.drop_duplicates()
                pool = pool.sample(frac=1, random_state=seed)
                cap = math.ceil(caps[split] * 3 / len(requested))
                selected.update(pool.head(cap))
    by_image = {image_id: frame.to_dict('records') for image_id, frame in annotations[annotations.ImageID.isin(selected)].groupby('ImageID')}
    def fetch(image_id):
        try:
            target = cache / 'images' / f'{image_id}.jpg'
            download_public(f'https://s3.amazonaws.com/open-images-dataset/validation/{image_id}.jpg', target, max_bytes=30_000_000)
            return image_id, target
        except Exception:
            return image_id, None
    paths = {}
    with ThreadPoolExecutor(max_workers=8) as executor:
        jobs = [executor.submit(fetch, image_id) for image_id in sorted(selected)]
        for future in tqdm(as_completed(jobs), total=len(jobs), desc='Open Images downloads'):
            image_id, path = future.result(); paths[image_id] = path
    output, counts, subtype_counts, ignored = [], Counter(), Counter(), Counter()
    rng = random.Random(seed)
    image_ids = sorted(selected); rng.shuffle(image_ids)
    per_subtype = {split: math.ceil(caps[split] / max(1, len(obstruction_names))) for split in caps}
    for image_id in tqdm(image_ids, desc='Open Images three-class crops'):
        path, split = paths[image_id], split_map[image_id]
        if path is None:
            ignored['download_failed'] += 1; continue
        try:
            with Image.open(path) as original:
                image = original.convert('RGB')
                objects = []
                for row in by_image[image_id]:
                    box = clean_bbox([float(row['XMin']) * image.width, float(row['YMin']) * image.height,
                                      float(row['XMax'] - row['XMin']) * image.width, float(row['YMax'] - row['YMin']) * image.height], image.width, image.height, min_size=1)
                    if box:
                        objects.append((row, box))
                for label in (1, 2):
                    candidates = [(row, box) for row, box in objects if row['label'] == label and not row['IsGroupOf'] and not row['IsDepiction']
                                  and (box[2] - box[0]) * (box[3] - box[1]) >= 400]
                    rng.shuffle(candidates)
                    for row, box in candidates:
                        if counts[split, label] >= caps[split]: break
                        if label == 2 and subtype_counts[split, row['name']] >= per_subtype[split]: continue
                        other = [b for item, b in objects if item['label'] == (2 if label == 1 else 1)]
                        crop_box = padded_box(box, image.width, image.height, rng)
                        if any(overlaps(crop_box, b) for b in other): crop_box = box
                        if any(overlaps(crop_box, b) for b in other):
                            ignored['mixed_vehicle_obstruction'] += 1; continue
                        output.append(save_status_crop(image, crop_box, crop_dir, label=label, source='openimages', group_id='oi_image_' + image_id,
                                                       split=split, source_id=image_id, source_sha=file_hash(str(path)), subtype=row['name']))
                        counts[split, label] += 1; subtype_counts[split, row['name']] += 1
                        break  # At most one positive of each status per source image.
                if counts[split, 0] < caps[split]:
                    # Exclude ALL annotated objects, not only cars, from background negatives.
                    box = negative_crop(image.width, image.height, [box for _, box in objects], rng)
                    if box:
                        output.append(save_status_crop(image, box, crop_dir, label=0, source='openimages', group_id='oi_image_' + image_id,
                                                       split=split, source_id=image_id, source_sha=file_hash(str(path)), subtype='annotation_free_background'))
                        counts[split, 0] += 1
        except (OSError, ValueError): ignored['unreadable_image'] += 1
    if not output:
        raise ValueError('Open Images produced no usable crops; check the network/downloads.')
    # Preserve attribution metadata for selected source images in the exported manifest.
    credits = pd.read_csv(download_public(OI_CREDITS, cache / 'credits.csv'))
    keep = [column for column in ['ImageID', 'OriginalURL', 'OriginalLandingURL', 'License', 'Author', 'Title'] if column in credits]
    result = pd.DataFrame(output).merge(credits[keep], left_on='source_id', right_on='ImageID', how='left')
    print('Open Images retained:', dict(counts), 'Skipped:', dict(ignored))
    return result


def perceptual_groups(paths, distance=4):
    """Group exact/near duplicate cone source photos before assigning any split."""
    hashes, parents = [], list(range(len(paths)))
    def root(index):
        while parents[index] != index:
            parents[index] = parents[parents[index]]; index = parents[index]
        return index
    for index, path in enumerate(paths):
        with Image.open(path) as image:
            pixels = np.asarray(image.convert('L').resize((9, 8), Image.Resampling.BILINEAR))
        bits = (pixels[:, 1:] > pixels[:, :-1]).reshape(-1)
        value = sum(int(bit) << n for n, bit in enumerate(bits))
        for previous, other in enumerate(hashes):
            if (value ^ other).bit_count() <= distance: parents[root(index)] = root(previous)
        hashes.append(value)
    return [f'tracon_scene_{root(index)}' for index in range(len(paths))]


def build_tracon(cache_dir, crop_dir, caps, seed=42):
    cache = Path(cache_dir) / 'tracon'
    archive = download_public(f'https://codeload.github.com/ikatsamenis/Cone-Detection/zip/{TRACON_COMMIT}', cache / 'tracon.zip', max_bytes=200_000_000)
    root = safe_extract_zip(archive, cache / 'extracted')
    paths = sorted(path for path in root.rglob('*') if path.suffix.lower() in {'.jpg', '.jpeg', '.png'} and '/images/' in path.as_posix())
    if not paths: raise ValueError('TraCon images were not found in the pinned archive.')
    groups = perceptual_groups(paths)
    frame = grouped_split(pd.DataFrame({'path': paths, 'group_id': groups}), seed=seed)
    output, counts = [], Counter(); rng = random.Random(seed)
    for row in frame.sample(frac=1, random_state=seed).to_dict('records'):
        if counts[row['split']] >= caps[row['split']]: continue
        path = Path(row['path'])
        label_path = Path(str(path).replace('/images/', '/labels/')).with_suffix('.txt')
        if not label_path.exists(): continue
        with Image.open(path) as original:
            image = original.convert('RGB')
            candidates = []
            for line in label_path.read_text().splitlines():
                values = line.split()
                if len(values) != 5 or values[0] != '0': raise ValueError('Unexpected TraCon class/YOLO label format.')
                cx, cy, width, height = map(float, values[1:])
                box = clean_bbox([(cx - width / 2) * image.width, (cy - height / 2) * image.height,
                                  width * image.width, height * image.height], image.width, image.height)
                if box: candidates.append(box)
            rng.shuffle(candidates)
            # One crop per original photo; cone context remains varied without duplicating a scene.
            if candidates:
                box = padded_box(candidates[0], image.width, image.height, rng, maximum=0.18)
                output.append(save_status_crop(image, box, crop_dir, label=2, source='tracon', group_id=row['group_id'], split=row['split'],
                                               source_id=path.name, source_sha=file_hash(str(path)), subtype='Traffic cone'))
                counts[row['split']] += 1
    print('TraCon retained by split:', dict(counts), 'Source-photo groups:', len(set(groups)))
    return pd.DataFrame(output)


def build_site_three(root, crop_dir, seed=42):
    root = Path(root); records = []
    labels = {name: index for index, name in enumerate(STATUS_NAMES)}
    for path in root.rglob('*'):
        if not path.is_file() or path.suffix.lower() not in {'.jpg', '.jpeg', '.png', '.webp'}: continue
        if path.parent.name.lower() not in labels: continue
        group = path.parent.parent
        if group.resolve() == root.resolve(): raise ValueError('Use day/session folders above vacant/occupied/blocked, not random adjacent-frame splits.')
        records.append(dict(path=path, label=labels[path.parent.name.lower()], group_id='site_' + str(group.relative_to(root))))
    if not records: raise ValueError('No site crops found. Use day01/vacant, day01/occupied and day01/blocked folders.')
    frame = grouped_split(pd.DataFrame(records), seed=seed); output = []
    for row in frame.to_dict('records'):
        with Image.open(row['path']) as original:
            image = ImageOps.exif_transpose(original).convert('RGB')
            if min(image.size) < 16: raise ValueError('A site crop is too small.')
            output.append(save_status_crop(image, (0, 0, image.width, image.height), crop_dir, label=row['label'], source='site',
                group_id=row['group_id'], split=row['split'], source_id=row['path'].name, source_sha=file_hash(str(row['path'])), subtype='site_labeled'))
    return pd.DataFrame(output)


def audit_three(frame):
    frame = frame.copy().reset_index(drop=True)
    if set(frame.label.unique()) != {0, 1, 2} or set(frame.split.unique()) != {'train', 'val', 'test'}:
        raise ValueError('Need explicit vacant=0, occupied=1, blocked=2 across train/val/test.')
    if frame.groupby('group_id').split.nunique().max() > 1: raise ValueError('A capture/scene group crosses splits.')
    conflicts = set(frame.groupby('crop_sha').label.nunique().loc[lambda x: x > 1].index)
    removed = frame[frame.crop_sha.isin(conflicts)].assign(reason='conflicting_labels')
    frame = frame[~frame.crop_sha.isin(conflicts)].copy()
    frame['_priority'] = frame.split.map({'test': 0, 'val': 1, 'train': 2})
    for column in ('source_sha', 'crop_sha'):
        preferred = frame.sort_values('_priority').drop_duplicates(column).set_index(column).split
        mask = frame.split != frame[column].map(preferred)
        removed = pd.concat([removed, frame[mask].assign(reason=f'{column}_cross_split')], ignore_index=True)
        frame = frame[~mask].copy()
    frame = frame.drop_duplicates('crop_sha').drop(columns='_priority').reset_index(drop=True)
    for split in ('train', 'val', 'test'):
        for source in [None] + (['site'] if 'site' in set(frame.source) else []):
            subset = frame[(frame.split == split) & ((frame.source == source) if source else True)]
            counts = subset.label.value_counts()
            if any(counts.get(label, 0) < 5 for label in range(3)):
                raise ValueError(f'{source or "combined"}/{split} needs five unique crops per class after audit. Add independent sessions with all three classes.')
    return frame, removed.drop(columns='_priority', errors='ignore')

In [ ]:
sources = []
if CONFIG['USE_PKLOT']:
    if CONFIG['PKLOT_ROOT']:
        pklot_root = Path(CONFIG['PKLOT_ROOT'])
    else:
        import kagglehub
        pklot_root = Path(kagglehub.dataset_download('ammarnassanalhajali/pklot-dataset'))
    sources.append(build_pklot(pklot_root, CROP_DIR, CONFIG['PKLOT_PER_CLASS'], CONFIG['SEED']))
if CONFIG['USE_OPEN_IMAGES']:
    sources.append(build_openimages(CACHE_DIR, CROP_DIR, CONFIG['OPENIMAGES_PER_CLASS'],
                                   CONFIG['VEHICLE_NAMES'], CONFIG['OBSTRUCTION_NAMES'], CONFIG['SEED']))
if CONFIG['USE_TRACON']:
    sources.append(build_tracon(CACHE_DIR, CROP_DIR, CONFIG['TRACON_CAPS'], CONFIG['SEED']))
for frame in sources:
    display(frame.groupby(['source', 'split', 'label']).size().rename('examples').reset_index())

## 4 · Optional site crops — recommended before deployment
Set `UPLOAD_SITE_ZIP=True` above, or set `SITE_ROOT` to an extracted folder. Use real **crops of the same fixed parking position**, matching the saved app box. Include independent day/night sessions, empty ground, vehicles of different colors, cones, boxes, and other relevant obstructions. Avoid unrelated ceiling and neighboring spaces in the box. Do not draw a different favorable crop for every image. A sack on the parking footprint can be blocked; a sack permanently outside it is background.

```text
site_crops/
  day01/
    vacant/      # Clear intended usable area, no vehicle.
    occupied/    # Vehicle occupies that area.
    blocked/     # Non-vehicle object obstructs that area.
  day02/
    vacant/...
    occupied/...
    blocked/...
  day03/
    vacant/...
    occupied/...
    blocked/...
```

Use **at least three independent recording days/sessions**, with all three classes in each session and at least five unique crops per class in every final split (a technical minimum, not enough to establish deployment quality). Keep adjacent frames and all clips from one session together. More independent sessions are better than many nearly identical frames. Reserve new sessions for a final site test. Do not reuse the old binary negative folders: cones/people may have been labeled vacant there. Do not label every non-car background as blocked.

In [ ]:
site_root = CONFIG['SITE_ROOT']
if CONFIG['UPLOAD_SITE_ZIP']:
    from google.colab import files
    uploaded = files.upload()
    archives = [name for name in uploaded if name.lower().endswith('.zip')]
    if len(archives) != 1: raise ValueError('Upload one ZIP with session/vacant, occupied, blocked folders.')
    site_root = str(safe_extract_zip(archives[0], CACHE_DIR / 'site_upload'))
if site_root:
    sources.append(build_site_three(site_root, CROP_DIR, CONFIG['SEED']))
else:
    print('SITE PERFORMANCE NOT MEASURED: public-data candidate only.')
if not sources: raise ValueError('Enable public datasets or supply site crops.')

## 5 · Audit and inspect labels before training
This stops when a class/split is missing. Single-class cone data is allowed as one source, but all three classes are required in the combined splits and in site splits if supplied. Exact duplicate photos cannot leak across splits. TraCon additionally groups similar source photos; grouping cannot establish that every related scene was found. Manually inspect backgrounds and positive object crops. Public objects fill their crop more than a small object in a wide parking view; site crops are needed to address that difference.

In [ ]:
manifest, excluded = audit_three(pd.concat(sources, ignore_index=True))
manifest.to_csv(RUN_DIR / 'manifest.csv', index=False)
excluded.to_csv(RUN_DIR / 'excluded_records.csv', index=False)
display(manifest.groupby(['source', 'split', 'label']).size().rename('examples').reset_index())
assert manifest.groupby('group_id').split.nunique().max() == 1
assert manifest.groupby('source_sha').split.nunique().max() == 1
assert manifest.groupby('crop_sha').split.nunique().max() == 1
train_examples = manifest[manifest.split == 'train']
n = min(6, int(train_examples.label.value_counts().min()))
fig, axes = plt.subplots(3, n, figsize=(3 * n, 8), squeeze=False)
for label, row_axes in enumerate(axes):
    examples = train_examples[train_examples.label == label].sample(n=n, random_state=CONFIG['SEED'])
    for axis, item in zip(row_axes, examples.to_dict('records')):
        with Image.open(item['path']) as image: axis.imshow(image)
        axis.set_title(f"{STATUS_NAMES[label]} · {item['source']}\n{item.get('subtype', '')}", fontsize=8)
        axis.axis('off')
fig.tight_layout(); plt.show()
print('Retained:', len(manifest), 'Excluded:', len(excluded))
print('Inspect Box and Traffic cone support in the table; download failures can reduce category coverage.')
if 'subtype' in manifest: display(manifest[manifest.label == 2].groupby(['split', 'subtype']).size().rename('blocked_examples').reset_index())

In [ ]:
SOURCES_TEXT = "# Dataset sources and reporting\n- PKLot: https://web.inf.ufpr.br/vri/databases/parking-lot-database/ ; mirror: https://www.kaggle.com/datasets/ammarnassanalhajali/pklot-dataset . Retains dataset terms/citation.\n- Open Images V7 class list and V5 validation boxes: https://storage.googleapis.com/openimages/web/download_v7.html . Selected source images retain their original licenses; OriginalURL/License/Author are carried into manifest.csv. Verify original image terms before redistribution.\n- TraCon: https://github.com/ikatsamenis/Cone-Detection ; paper https://arxiv.org/abs/2205.11830 . Pinned commit 7ac4af371e826b5124a9b4defaa3327bc817817e. Cite the authors. No explicit dataset license file was found in that pinned repository; confirm permission before redistributing its images or commercial use. Raw images are not bundled in the model export.\n- EfficientNet inputs: https://keras.io/api/applications/efficientnet/ ; fine-tuning: https://keras.io/guides/transfer_learning/ . RGB 0..255, preprocessing inside EfficientNet.\n\nPKLot is regrouped by capture day. The Open Images validation SOURCE POOL is repartitioned into this experiment's own train/validation/test groups by image, not reported as its official benchmark. TraCon source photos are regrouped with perceptual near-duplicate grouping before splitting, ignoring its original split. This reduces leakage but cannot prove all related scenes were found. Keep your own capture sessions together.\nPublic object and annotation-free background crops are bootstrap proxies. An annotation-free patch is not verified parking ground; an object in an image does not prove it obstructs a parking space. Site-labeled fixed-box crops establish that distinction. Do not reuse binary COCO 'unoccupied' crops: those can contain people/objects that now need a different label.\nReport the new run's full holdout scores, class precision/recall/support, per-source scores, and coverage of the review filter. Do not reuse the old S2 score, tune settings against test errors, or describe public performance as site accuracy. The three-class argmax is separate from the app's 0.80 review threshold. Confidence is not verified accuracy. Any site data included in previous model training must not be used as a new independent test set.\n"
(RUN_DIR / 'SOURCES.md').write_text(SOURCES_TEXT)

## 6 · EfficientNetB0 and datasets
RGB float32 inputs remain **0–255**; EfficientNet includes scaling. A three-unit softmax replaces the binary head. Augmentation applies during training only. Training is balanced by class, then by available source within each class. Validation checkpoint loss is class/source balanced. Final holdout accuracy is ordinary unweighted accuracy, reported separately by source. No augmentation or oversampling of the test set.

In [ ]:
"""EfficientNetB0-only training/export helpers embedded into the Colab notebook."""
import hashlib
import json
import math
from pathlib import Path
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import (accuracy_score, average_precision_score, balanced_accuracy_score,
                             brier_score_loss, classification_report, confusion_matrix,
                             f1_score, precision_score, recall_score, roc_auc_score)


def balanced_training_frame(frame, source_weights, max_samples=16000, seed=42):
    """Balance classes AND source contributions; do not let PKLot drown out local data."""
    sources = sorted(frame.source.unique())
    total_weight = sum(source_weights.get(source, 1.0) for source in sources)
    target = min(max_samples, max(len(frame), 128))
    sampled = []
    for source in sources:
        weight = source_weights.get(source, 1.0)
        count = max(1, round(target * weight / total_weight / 2))
        for label in (0, 1):
            pool = frame[(frame.source == source) & (frame.label == label)]
            if pool.empty:
                raise ValueError(f'Training source {source} has no class {label}; collect more recording groups.')
            sampled.append(pool.sample(n=count, replace=count > len(pool), random_state=seed + label))
    return pd.concat(sampled, ignore_index=True).sample(frac=1, random_state=seed).reset_index(drop=True)


def validation_weights(frame, source_weights):
    weights = np.zeros(len(frame), dtype=np.float32)
    for source in frame.source.unique():
        for label in (0, 1):
            mask = (frame.source == source) & (frame.label == label)
            count = int(mask.sum())
            if count:
                weights[mask.to_numpy()] = source_weights.get(source, 1.0) / count / 2
    return weights / weights.mean()


def make_dataset(frame, batch_size=32, training=False, weights=None, seed=42):
    paths = frame.path.astype(str).to_numpy()
    labels = frame.label.to_numpy(dtype=np.float32)
    if weights is None:
        dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    else:
        dataset = tf.data.Dataset.from_tensor_slices((paths, labels, np.asarray(weights, dtype=np.float32)))
    if training:
        dataset = dataset.shuffle(min(len(frame), 4096), seed=seed, reshuffle_each_iteration=True)
    def decode(path, label, *sample_weight):
        image = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
        image.set_shape([None, None, 3])
        # Same RGB, bilinear resize, float32 and 0–255 contract as ml/worker.py.
        image = tf.image.resize(tf.cast(image, tf.float32), (224, 224), method='bilinear')
        return (image, label, sample_weight[0]) if sample_weight else (image, label)
    dataset = dataset.map(decode, num_parallel_calls=tf.data.AUTOTUNE)
    options = tf.data.Options()
    options.experimental_deterministic = not training
    return dataset.with_options(options).batch(batch_size).prefetch(tf.data.AUTOTUNE)


def build_vehicle_model(weights='imagenet', seed=42):
    inputs = tf.keras.Input((224, 224, 3), dtype=tf.float32, name='parking_crop_rgb')
    augmentation = tf.keras.Sequential([
        tf.keras.layers.RandomFlip('horizontal', seed=seed),
        tf.keras.layers.RandomRotation(0.025, fill_mode='reflect', seed=seed + 1),
        tf.keras.layers.RandomTranslation(0.04, 0.04, fill_mode='reflect', seed=seed + 2),
        tf.keras.layers.RandomZoom(0.10, fill_mode='reflect', seed=seed + 3),
        tf.keras.layers.RandomContrast(0.20, seed=seed + 4),
        tf.keras.layers.RandomBrightness(0.20, value_range=(0, 255), seed=seed + 5),
    ], name='training_augmentation')
    base = tf.keras.applications.EfficientNetB0(include_top=False, weights=weights, input_shape=(224, 224, 3), name='efficientnetb0')
    base.trainable = False
    x = base(augmentation(inputs), training=False)
    x = tf.keras.layers.GlobalAveragePooling2D(name='parking_pool')(x)
    x = tf.keras.layers.Dropout(0.30, seed=seed + 6, name='training_dropout')(x)
    output = tf.keras.layers.Dense(1, activation='sigmoid', name='vehicle_probability')(x)
    return tf.keras.Model(inputs, output, name='ParkVision_EfficientNetB0_Training')


def compile_vehicle_model(model, learning_rate):
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate, clipnorm=1.0),
                  loss=tf.keras.losses.BinaryCrossentropy(label_smoothing=0.02),
                  metrics=[tf.keras.metrics.BinaryAccuracy(name='accuracy', threshold=0.5),
                           tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall'),
                           tf.keras.metrics.AUC(name='auc')], weighted_metrics=[])


def unfreeze_top_blocks(model):
    base = model.get_layer('efficientnetb0')
    base.trainable = True
    for layer in base.layers:
        layer.trainable = layer.name.startswith(('block6', 'block7', 'top_conv')) and not isinstance(layer, tf.keras.layers.BatchNormalization)
    return sum(layer.trainable for layer in base.layers)


def train_stages(train_dataset, validation_dataset, run_dir, head_epochs=5, fine_epochs=10, seed=42, weights='imagenet'):
    run_dir = Path(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)
    def callbacks(stage):
        return [tf.keras.callbacks.ModelCheckpoint(str(run_dir / f'best_{stage}.keras'), monitor='val_loss', save_best_only=True),
                tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
                tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', patience=2, factor=0.5, min_lr=1e-7),
                tf.keras.callbacks.CSVLogger(str(run_dir / f'{stage}_history.csv')),
                tf.keras.callbacks.TerminateOnNaN()]
    model = build_vehicle_model(weights=weights, seed=seed)
    compile_vehicle_model(model, 1e-3)
    print('Stage 1: train the binary classification head; convolutional base frozen.')
    head = model.fit(train_dataset, validation_data=validation_dataset, epochs=head_epochs, callbacks=callbacks('head'))
    model = tf.keras.models.load_model(run_dir / 'best_head.keras', compile=False)
    print('Stage 2: fine-tune final EfficientNet blocks; BatchNorm stays frozen.')
    print('Trainable backbone layers:', unfreeze_top_blocks(model))
    compile_vehicle_model(model, 1e-5)
    fine = model.fit(train_dataset, validation_data=validation_dataset, epochs=fine_epochs, callbacks=callbacks('finetune'))
    candidates = [('head', min(head.history['val_loss'])), ('finetune', min(fine.history['val_loss']))]
    selected_stage, best_loss = min(candidates, key=lambda item: item[1])
    # Selection uses validation loss ONLY. Test results cannot change this choice.
    best = tf.keras.models.load_model(run_dir / f'best_{selected_stage}.keras', compile=False)
    info = {'selected_stage': selected_stage, 'validation_loss': float(best_loss),
            'head_epochs_run': len(head.history['loss']), 'fine_epochs_run': len(fine.history['loss']),
            'selection_metric': 'source-balanced validation loss'}
    return best, info, {'head': head.history, 'finetune': fine.history}


def predict_frame(model, frame, batch_size=32):
    dataset = make_dataset(frame, batch_size=batch_size)
    probabilities = np.asarray(model.predict(dataset, verbose=0)).reshape(-1)
    if len(probabilities) != len(frame) or not np.isfinite(probabilities).all():
        raise ValueError('Prediction count/values are invalid.')
    return probabilities


def binary_metrics(labels, probabilities):
    labels = np.asarray(labels, dtype=int)
    predictions = (np.asarray(probabilities) >= 0.5).astype(int)
    both_classes = len(np.unique(labels)) == 2
    return {'examples': int(len(labels)), 'unoccupied_examples': int((labels == 0).sum()),
            'occupied_examples': int((labels == 1).sum()), 'accuracy': float(accuracy_score(labels, predictions)),
            'balanced_accuracy': float(balanced_accuracy_score(labels, predictions)) if both_classes else None,
            'precision_occupied': float(precision_score(labels, predictions, zero_division=0)),
            'recall_occupied': float(recall_score(labels, predictions, zero_division=0)),
            'f1_occupied': float(f1_score(labels, predictions, zero_division=0)),
            'roc_auc': float(roc_auc_score(labels, probabilities)) if both_classes else None,
            'average_precision': float(average_precision_score(labels, probabilities)) if both_classes else None,
            'brier_score': float(brier_score_loss(labels, probabilities)),
            'confusion_matrix': confusion_matrix(labels, predictions, labels=[0, 1]).tolist(),
            'predicted_unoccupied': int((predictions == 0).sum()), 'predicted_occupied': int((predictions == 1).sum())}


def evaluate_holdout(model, manifest, run_dir, batch_size=32):
    run_dir = Path(run_dir)
    test = manifest[manifest.split == 'test'].reset_index(drop=True).copy()
    test['p_occupied'] = predict_frame(model, test, batch_size)
    test['prediction'] = (test.p_occupied >= 0.5).astype(int)
    report = {'threshold': 0.5, 'overall': binary_metrics(test.label, test.p_occupied), 'by_source': {}}
    for source, frame in test.groupby('source'):
        report['by_source'][source] = binary_metrics(frame.label, frame.p_occupied)
    print(classification_report(test.label, test.prediction, labels=[0, 1], target_names=['unoccupied', 'occupied'], zero_division=0))
    print('Per-source holdout results:')
    display_columns = ['examples', 'accuracy', 'precision_occupied', 'recall_occupied', 'f1_occupied']
    print(pd.DataFrame(report['by_source']).T[display_columns].to_string())
    if (test.prediction == test.prediction.iloc[0]).all():
        print('WARNING: The model predicted one class for the entire test set. Inspect data and do not claim it is validated.')
    test.to_csv(run_dir / 'test_predictions.csv', index=False)
    test[test.label != test.prediction].to_csv(run_dir / 'test_errors.csv', index=False)
    (run_dir / 'metrics.json').write_text(json.dumps(report, indent=2))
    fig, axis = plt.subplots(figsize=(5, 4))
    matrix = np.asarray(report['overall']['confusion_matrix'])
    axis.imshow(matrix, cmap='Blues')
    for (row, column), value in np.ndenumerate(matrix):
        axis.text(column, row, str(value), ha='center', va='center', color='white' if value > matrix.max() / 2 else 'black')
    axis.set(xticks=[0, 1], yticks=[0, 1], xticklabels=['unoccupied', 'occupied'], yticklabels=['unoccupied', 'occupied'],
             xlabel='Predicted', ylabel='Actual', title='Untouched holdout · EfficientNetB0')
    fig.tight_layout(); fig.savefig(run_dir / 'confusion_matrix.png', dpi=160); plt.show()
    return report, test


def inference_only_model(trained_model):
    """Reuse learned weights, discard augmentation/dropout; no custom layers needed."""
    inputs = tf.keras.Input((224, 224, 3), dtype=tf.float32, name='parking_crop_rgb')
    x = trained_model.get_layer('efficientnetb0')(inputs, training=False)
    x = trained_model.get_layer('parking_pool')(x)
    outputs = trained_model.get_layer('vehicle_probability')(x)
    return tf.keras.Model(inputs, outputs, name='ParkVision_EfficientNetB0_VehiclePresence')


def export_vehicle_model(trained_model, manifest, metrics, training_info, run_dir, config):
    run_dir = Path(run_dir)
    export_dir = run_dir / 'export'
    export_dir.mkdir(exist_ok=True)
    model = inference_only_model(trained_model)
    validation = manifest[manifest.split == 'val'].head(16)
    sample = next(iter(make_dataset(validation, batch_size=16)))[0]
    np.testing.assert_allclose(trained_model(sample, training=False).numpy(), model(sample, training=False).numpy(), rtol=1e-5, atol=1e-6)
    model_path = export_dir / 'parkvision_efficientnetb0.keras'
    model.save(model_path)
    loaded = tf.keras.models.load_model(model_path, compile=False)
    np.testing.assert_allclose(model(sample, training=False).numpy(), loaded(sample, training=False).numpy(), rtol=1e-5, atol=1e-6)
    local = metrics['by_source'].get('site')
    metadata = {'format_version': 1, 'model_name': model.name, 'model_version': run_dir.name,
                'class_names': ['unoccupied', 'occupied'], 'app_statuses': ['vacant', 'occupied'],
                'semantic_target': 'target vehicle present inside the supplied crop',
                'vehicle_categories': list(config['VEHICLE_CATEGORIES']), 'occupied_threshold': 0.5,
                'input_size': [224, 224], 'color_mode': 'RGB', 'input_dtype': 'float32', 'input_range': [0, 255],
                'resize_method': 'bilinear', 'preprocessing_inside_model': True,
                'tensorflow_version': tf.__version__, 'keras_version': tf.keras.__version__,
                'sha256': hashlib.sha256(model_path.read_bytes()).hexdigest(), 'training': training_info,
                'test_metrics': metrics, 'site_test_evaluated': bool(local and local['unoccupied_examples'] >= 5 and local['occupied_examples'] >= 5),
                'deployment_status': 'candidate_requires_site_validation',
                'save_reload_parity_passed': True, 'augmentation_removed': True}
    metadata_path = export_dir / 'parkvision_efficientnetb0.metadata.json'
    metadata_path.write_text(json.dumps(metadata, indent=2))
    (export_dir / 'README.txt').write_text(
        'ParkVision EfficientNetB0 binary vehicle-presence classifier\n'
        '0 = unoccupied (no target vehicle); 1 = occupied (target vehicle).\n'
        'No target vehicle does not mean physically unobstructed/parkable.\n'
        'This export is a candidate; inspect metrics and held-out site results.\n'
        'Copy BOTH model and metadata into ParkVision/ml/models, backing up the old pair.\n'
        'Restart ParkVision after replacing model files. The input stays RGB 224x224 float32, 0..255.\n'
        'Do not divide by 255; preprocessing is inside EfficientNet.\n'
        'The app review threshold is separate from the binary decision threshold (0.5).\n'
    )
    archive = run_dir / 'parkvision_efficientnetb0_export.zip'
    with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
        for path in export_dir.iterdir():
            bundle.write(path, path.name)
        for name in ['metrics.json', 'test_predictions.csv', 'test_errors.csv', 'confusion_matrix.png', 'manifest.csv', 'training_info.json']:
            if (run_dir / name).exists():
                bundle.write(run_dir / name, name)
    return model, metadata, archive

In [ ]:
"""EfficientNetB0 vacant/vehicle/obstruction training and app-compatible export."""
import hashlib
import json
from pathlib import Path
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_recall_fscore_support


THREE_CLASSES = ['vacant', 'occupied', 'blocked']


def balanced_three(frame, source_weights, max_samples=16000, seed=42):
    draws = max(3, min(max_samples, max(len(frame), 128))) // 3
    parts = []
    for label in range(3):
        subset = frame[frame.label == label]
        if subset.empty: raise ValueError(f'Training class {THREE_CLASSES[label]} is missing.')
        sources = sorted(subset.source.unique())
        total = sum(source_weights.get(source, 1.0) for source in sources)
        for source in sources:
            pool = subset[subset.source == source]
            count = max(1, round(draws * source_weights.get(source, 1.0) / total))
            parts.append(pool.sample(n=count, replace=count > len(pool), random_state=seed + label))
    return pd.concat(parts, ignore_index=True).sample(frac=1, random_state=seed).reset_index(drop=True)


def three_validation_weights(frame, source_weights):
    result = np.zeros(len(frame), dtype=np.float32)
    for label in range(3):
        subset = frame[frame.label == label]
        sources = sorted(subset.source.unique())
        if not sources: raise ValueError('Validation needs all three classes.')
        total = sum(source_weights.get(source, 1.0) for source in sources)
        for source in sources:
            mask = (frame.label == label) & (frame.source == source)
            result[mask.to_numpy()] = source_weights.get(source, 1.0) / total / int(mask.sum()) / 3
    return result / result.mean()


def build_three_model(weights='imagenet', seed=42, starting_model=None):
    template = build_vehicle_model(weights=None if starting_model else weights, seed=seed)
    if starting_model:
        previous = tf.keras.models.load_model(starting_model, compile=False)
        if tuple(previous.input_shape[1:]) != (224, 224, 3): raise ValueError('Warm-start model must accept 224x224 RGB.')
        template.get_layer('efficientnetb0').set_weights(previous.get_layer('efficientnetb0').get_weights())
    probabilities = tf.keras.layers.Dense(3, activation='softmax', name='parking_status_probabilities')(template.get_layer('training_dropout').output)
    return tf.keras.Model(template.input, probabilities, name='ParkVision_EfficientNetB0_ThreeClass_Training')


def compile_three(model, rate):
    model.compile(optimizer=tf.keras.optimizers.Adam(rate, clipnorm=1.0),
                  loss=tf.keras.losses.SparseCategoricalCrossentropy(),
                  metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name='accuracy')], weighted_metrics=[])


def fit_three(train_ds, val_ds, run_dir, head_epochs=5, fine_epochs=10, seed=42, weights='imagenet', starting_model=None):
    run_dir = Path(run_dir); run_dir.mkdir(parents=True, exist_ok=True)
    def callbacks(stage):
        return [tf.keras.callbacks.ModelCheckpoint(str(run_dir / f'best_{stage}.keras'), monitor='val_loss', save_best_only=True),
                tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
                tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', patience=2, factor=0.5, min_lr=1e-7),
                tf.keras.callbacks.CSVLogger(str(run_dir / f'{stage}_history.csv')), tf.keras.callbacks.TerminateOnNaN()]
    model = build_three_model(weights, seed, starting_model)
    compile_three(model, 1e-3)
    print('Stage 1: train the three-class head.')
    head = model.fit(train_ds, validation_data=val_ds, epochs=head_epochs, callbacks=callbacks('head'))
    model = tf.keras.models.load_model(run_dir / 'best_head.keras', compile=False)
    print('Stage 2: fine-tune EfficientNetB0 blocks 6/7; BatchNorm remains frozen.')
    print('Trainable backbone layers:', unfreeze_top_blocks(model))
    compile_three(model, 1e-5)
    fine = model.fit(train_ds, validation_data=val_ds, epochs=fine_epochs, callbacks=callbacks('finetune'))
    stage, loss = min([('head', min(head.history['val_loss'])), ('finetune', min(fine.history['val_loss']))], key=lambda x: x[1])
    info = dict(selected_stage=stage, validation_loss=float(loss), selection_metric='class/source-balanced validation loss',
                head_epochs_run=len(head.history['loss']), fine_epochs_run=len(fine.history['loss']),
                initialization='supplied EfficientNetB0 backbone' if starting_model else 'ImageNet EfficientNetB0' if weights == 'imagenet' else 'random initialization (mechanics test only)')
    return tf.keras.models.load_model(run_dir / f'best_{stage}.keras', compile=False), info, dict(head=head.history, finetune=fine.history)


def three_metrics(labels, probabilities):
    labels = np.asarray(labels, dtype=int); probabilities = np.asarray(probabilities)
    if probabilities.shape != (len(labels), 3) or not np.isfinite(probabilities).all(): raise ValueError('Invalid three-class prediction array.')
    if (probabilities < 0).any() or (probabilities > 1).any() or not np.allclose(probabilities.sum(axis=1), 1, atol=1e-3): raise ValueError('Expected softmax probabilities.')
    predicted = probabilities.argmax(axis=1)
    precision, recall, f1, support = precision_recall_fscore_support(labels, predicted, labels=[0, 1, 2], zero_division=0)
    present = sorted(set(labels.tolist()))
    return dict(examples=len(labels), accuracy=float(accuracy_score(labels, predicted)),
                macro_f1=float(f1_score(labels, predicted, labels=present, average='macro', zero_division=0)),
                macro_f1_classes=[THREE_CLASSES[index] for index in present],
                confusion_matrix=confusion_matrix(labels, predicted, labels=[0, 1, 2]).tolist(),
                by_class={name: dict(precision=float(precision[index]), recall=float(recall[index]), f1=float(f1[index]),
                                     support=int(support[index])) for index, name in enumerate(THREE_CLASSES)})


def evaluate_three(model, manifest, run_dir, batch_size=32, review_threshold=0.8):
    run_dir = Path(run_dir)
    test = manifest[manifest.split == 'test'].copy().reset_index(drop=True)
    probabilities = np.asarray(model.predict(make_dataset(test, batch_size), verbose=0))
    report = dict(class_order=THREE_CLASSES, overall=three_metrics(test.label, probabilities), by_source={})
    test['prediction'] = probabilities.argmax(axis=1); test['confidence'] = probabilities.max(axis=1)
    test['needs_review'] = test.confidence < review_threshold
    for index, name in enumerate(THREE_CLASSES): test[f'p_{name}'] = probabilities[:, index]
    for source, frame in test.groupby('source'):
        report['by_source'][source] = three_metrics(frame.label, probabilities[frame.index])
    if 'subtype' in test:
        report['positive_subtype_checks'] = [dict(subtype=str(subtype), examples=len(frame),
            correct_class_fraction=float((frame.label == frame.prediction).mean()))
            for subtype, frame in test.dropna(subset=['subtype']).groupby('subtype')]
    accepted = ~test.needs_review
    report['review_filter'] = dict(threshold=review_threshold, accepted=int(accepted.sum()), needs_review=int((~accepted).sum()),
        accepted_accuracy=float((test[accepted].label == test[accepted].prediction).mean()) if accepted.any() else None,
        coverage=float(accepted.mean()))
    print(classification_report(test.label, test.prediction, labels=[0, 1, 2], target_names=THREE_CLASSES, zero_division=0))
    print('All-example metrics:', report['overall'])
    print('Review coverage:', report['review_filter'])
    print('Site metrics:', report['by_source'].get('site', 'NOT MEASURED'))
    if test.prediction.nunique() < 3: print('WARNING: at least one class was never predicted. Inspect the confusion matrix and do not claim the task is solved.')
    test.to_csv(run_dir / 'test_predictions.csv', index=False)
    test[test.label != test.prediction].to_csv(run_dir / 'test_errors.csv', index=False)
    (run_dir / 'metrics.json').write_text(json.dumps(report, indent=2))
    fig, ax = plt.subplots(figsize=(6, 5)); matrix = np.asarray(report['overall']['confusion_matrix'])
    ax.imshow(matrix, cmap='Blues')
    for (row, column), value in np.ndenumerate(matrix): ax.text(column, row, str(value), ha='center', va='center')
    ax.set(xticks=[0, 1, 2], yticks=[0, 1, 2], xticklabels=THREE_CLASSES, yticklabels=THREE_CLASSES,
           xlabel='Predicted', ylabel='Actual', title='Untouched three-class holdout')
    fig.tight_layout(); fig.savefig(run_dir / 'confusion_matrix.png', dpi=160); plt.show()
    return report, test


def inference_three(trained):
    inputs = tf.keras.Input((224, 224, 3), dtype=tf.float32, name='parking_crop_rgb')
    x = trained.get_layer('efficientnetb0')(inputs, training=False)
    x = trained.get_layer('parking_pool')(x)
    outputs = trained.get_layer('parking_status_probabilities')(x)
    return tf.keras.Model(inputs, outputs, name='ParkVision_EfficientNetB0_ThreeClass')


def export_three(trained, manifest, metrics, info, run_dir, config):
    run_dir = Path(run_dir); folder = run_dir / 'export'; folder.mkdir(exist_ok=True)
    model = inference_three(trained)
    validation = manifest[manifest.split == 'val'].head(12)
    images = next(iter(make_dataset(validation, batch_size=12)))[0]
    np.testing.assert_allclose(trained(images, training=False).numpy(), model(images, training=False).numpy(), rtol=1e-5, atol=1e-6)
    path = folder / 'parkvision_efficientnetb0.keras'; model.save(path)
    loaded = tf.keras.models.load_model(path, compile=False)
    np.testing.assert_allclose(model(images, training=False).numpy(), loaded(images, training=False).numpy(), rtol=1e-5, atol=1e-6)
    coverage = manifest.groupby(['source', 'split', 'label']).size().rename('examples').reset_index().to_dict('records')
    metadata = dict(format_version=2, task='parking_three_class', model_name=model.name, model_version=run_dir.name,
        class_names=THREE_CLASSES, app_statuses=THREE_CLASSES, output_activation='softmax',
        input_size=[224, 224], color_mode='RGB', input_dtype='float32', input_range=[0, 255], resize_method='bilinear', preprocessing_inside_model=True,
        sha256=hashlib.sha256(path.read_bytes()).hexdigest(), tensorflow_version=tf.__version__, keras_version=tf.keras.__version__,
        vehicle_categories=config.get('VEHICLE_NAMES', []), obstruction_categories=config.get('OBSTRUCTION_NAMES', []) + ['Traffic cone'],
        label_policy='blocked when a non-vehicle object obstructs the intended usable area; otherwise occupied if a vehicle occupies it; otherwise vacant',
        public_data_limitation='public object/background crops are bootstrap proxies, not verified parking-ground occupancy labels',
        deployment_status='candidate_requires_site_validation', site_test_evaluated='site' in metrics['by_source'],
        training=info, test_metrics=metrics, data_coverage=coverage, save_reload_parity_passed=True, augmentation_removed=True)
    (folder / 'parkvision_efficientnetb0.metadata.json').write_text(json.dumps(metadata, indent=2))
    (folder / 'README.txt').write_text(
        'ParkVision three-class EfficientNetB0 candidate\n0 vacant; 1 occupied by vehicle; 2 blocked by non-vehicle obstruction.\n'
        'Uses the same saved parking-region crops. RGB float32 224x224, values 0..255; no extra /255.\n'
        'Back up the installed model pair, then copy BOTH .keras and .metadata.json into ml/models and restart ParkVision. No SQL reset.\n'
        'Three-class support must be present in ml/worker.py; binary-only workers cannot load this output.\n'
        'Review all-example and per-class metrics, including cone/Box support and independent site data. Public scores are not site accuracy.\n'
        'The export is a candidate; arbitrary obstructions and physical parkability are not proven.\n')
    archive = run_dir / 'parkvision_three_class_export.zip'
    with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
        for item in folder.iterdir(): bundle.write(item, item.name)
        for name in ['metrics.json', 'manifest.csv', 'excluded_records.csv', 'test_predictions.csv', 'test_errors.csv', 'training_info.json',
                     'config.json', 'confusion_matrix.png', 'training_curves.png', 'SOURCES.md']:
            if (run_dir / name).exists(): bundle.write(run_dir / name, name)
    return model, metadata, archive

In [ ]:
starting_model = None
if CONFIG['WARM_START_UPLOAD']:
    from google.colab import files
    uploaded = files.upload()
    paths = [Path(name) for name in uploaded]
    if len(paths) != 1: raise ValueError('Upload one previous .keras model or export ZIP.')
    if paths[0].suffix.lower() == '.zip':
        extracted = safe_extract_zip(paths[0], CACHE_DIR / 'warm_start')
        paths = list(extracted.rglob('parkvision_efficientnetb0.keras'))
    if len(paths) != 1 or paths[0].suffix != '.keras': raise ValueError('Could not find the previous .keras model.')
    starting_model = str(paths[0])
train_frame = manifest[manifest.split == 'train'].reset_index(drop=True)
val_frame = manifest[manifest.split == 'val'].reset_index(drop=True)
sampled = balanced_three(train_frame, CONFIG['SOURCE_WEIGHTS'], CONFIG['MAX_TRAIN_SAMPLES_PER_EPOCH'], CONFIG['SEED'])
train_ds = make_dataset(sampled, CONFIG['BATCH_SIZE'], training=True, seed=CONFIG['SEED'])
val_ds = make_dataset(val_frame, CONFIG['BATCH_SIZE'], weights=three_validation_weights(val_frame, CONFIG['SOURCE_WEIGHTS']))
images, labels = next(iter(train_ds))
assert images.shape[1:] == (224, 224, 3)
assert 0 <= float(tf.reduce_min(images)) <= float(tf.reduce_max(images)) <= 255
display(sampled.groupby(['source', 'label']).size().rename('training_draws').reset_index())

## 7 · Train the head, then fine-tune
Head training uses a frozen ImageNet backbone by default. Stage 2 unfreezes blocks 6/7 at a smaller learning rate; BatchNorm remains frozen. Save best checkpoints on Drive. Choose the candidate by validation loss only. Do not interrupt a working runtime to export, or restart it after training before saving.

In [ ]:
best_model, training_info, histories = fit_three(train_ds, val_ds, RUN_DIR,
    head_epochs=CONFIG['HEAD_EPOCHS'], fine_epochs=CONFIG['FINE_TUNE_EPOCHS'],
    seed=CONFIG['SEED'], weights='imagenet', starting_model=starting_model)
(RUN_DIR / 'training_info.json').write_text(json.dumps(training_info, indent=2))
(RUN_DIR / 'histories.json').write_text(json.dumps(histories, indent=2))
print(training_info)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for stage, history in histories.items():
    for key, axis in [('loss', axes[0]), ('accuracy', axes[1])]:
        axis.plot(history[key], label=f'{stage} train')
        axis.plot(history[f'val_{key}'], '--', label=f'{stage} validation')
for axis in axes: axis.set_xlabel('Epoch within stage'); axis.legend()
axes[0].set_title('Class/source-balanced validation loss'); axes[1].set_title('Unweighted accuracy')
fig.tight_layout(); fig.savefig(RUN_DIR / 'training_curves.png', dpi=160); plt.show()

## 8 · Untouched holdout evaluation
Prediction = highest of three scores. Report **all-example** accuracy/macro F1, each class's precision/recall/support, source-specific results, cone/Box checks, and the confusion matrix. The 80% publication filter is reported separately so reviewed/skipped mistakes do not disappear from the headline score. If site data is absent, site accuracy is NOT MEASURED. The source mix affects overall accuracy; public and local results are not interchangeable. Test errors are for reporting, not tuning this run.

In [ ]:
metrics, test_predictions = evaluate_three(best_model, manifest, RUN_DIR, CONFIG['BATCH_SIZE'], CONFIG['REVIEW_THRESHOLD'])
display(pd.DataFrame(metrics['overall']['by_class']).T)
display(pd.DataFrame({source: result['accuracy'] for source, result in metrics['by_source'].items()}, index=['accuracy']).T)
if 'positive_subtype_checks' in metrics: display(pd.DataFrame(metrics['positive_subtype_checks']))
errors = test_predictions[test_predictions.label != test_predictions.prediction].sort_values('confidence', ascending=False).head(12)
if len(errors):
    fig, axes = plt.subplots(3, 4, figsize=(12, 9))
    for axis in axes.ravel(): axis.axis('off')
    for axis, item in zip(axes.ravel(), errors.to_dict('records')):
        with Image.open(item['path']) as image: axis.imshow(image)
        axis.set_title(f"true={THREE_CLASSES[item['label']]}\npred={THREE_CLASSES[item['prediction']]} ({item['confidence']:.2f})", fontsize=8)
    fig.tight_layout(); fig.savefig(RUN_DIR / 'hard_errors.png', dpi=160); plt.show()

## 9 · Export and download
Native `.keras` export removes augmentation/dropout and checks output parity before and after saving/reloading. **Do not use `tf.saved_model.save`.** The model and checksum metadata must stay together. Exporting does not mean the candidate passed site validation.

In [ ]:
inference_model, metadata, export_zip = export_three(best_model, manifest, metrics, training_info, RUN_DIR, CONFIG)
print('Export:', export_zip)
print('Model:', metadata['model_name'], 'SHA256:', metadata['sha256'])
print('Site test evaluated:', metadata['site_test_evaluated'])
from google.colab import files
files.download(str(export_zip))
EXPORT_DONE_THIS_SESSION = True

## 10 · Install in ParkVision
1. Unzip `parkvision_three_class_export.zip` and review `metrics.json`, confusion matrix and error examples.
2. Back up the app's existing **model and metadata pair** outside `ml/models/`.
3. Copy **both** `parkvision_efficientnetb0.keras` and `parkvision_efficientnetb0.metadata.json` into `ParkVision/ml/models/`.
4. Restart `npm run dev`. The workspace should report three-class obstruction support and model name `ParkVision_EfficientNetB0_ThreeClass` with this export's checksum prefix.
5. Use the **same saved parking boxes**. Test clear/vehicle/cone/box examples. Publish accepted results; blocked should produce a slot warning/alert and never count as available. Later accepted clear/vehicle observations resolve the alert. Low class scores remain Needs review.

**Keep Supabase and all existing regions. No SQL reset is needed.** Older binary-only app workers must first be updated to three-class support. The app still uses a separate minimum class score of 0.80 for publication; this is not a verified accuracy guarantee. Cones outside the parking footprint should not be labeled blocked. New obstruction types need representative training and independent testing.

## 11 · Optional photo and box test
Set `RUN_MANUAL_IMAGE_TEST=True`, upload one photo and edit the normalized `TEST_BOXES`. These examples are illustrative. Test data is not used for training. Keep the same parking box when a car or object leaves. A correct single photo is a sanity check, not measured deployment accuracy.

In [ ]:
if CONFIG['RUN_MANUAL_IMAGE_TEST']:
    from google.colab import files
    uploaded = files.upload()
    image = ImageOps.exif_transpose(Image.open(next(iter(uploaded)))).convert('RGB')
    TEST_BOXES = [dict(name='left', x=0.08, y=0.45, width=0.38, height=0.50),
                  dict(name='right', x=0.46, y=0.40, width=0.53, height=0.48)]
    crops = []
    for box in TEST_BOXES:
        bounds = clean_bbox([box['x']*image.width, box['y']*image.height, box['width']*image.width, box['height']*image.height], image.width, image.height)
        if bounds is None: raise ValueError('Invalid test rectangle.')
        crops.append(tf.image.resize(np.asarray(image.crop(bounds), dtype=np.float32), (224, 224)))
    probabilities = np.asarray(inference_model(tf.stack(crops), training=False))
    fig, axes = plt.subplots(1, len(crops), figsize=(5*len(crops), 4), squeeze=False)
    for axis, box, crop, scores in zip(axes.ravel(), TEST_BOXES, crops, probabilities):
        winner = int(scores.argmax())
        axis.imshow(np.clip(crop.numpy(), 0, 255).astype('uint8')); axis.axis('off')
        axis.set_title(f"{box['name']}: {THREE_CLASSES[winner]} {scores[winner]:.1%}\n" + ('Needs review' if scores[winner] < CONFIG['REVIEW_THRESHOLD'] else 'Above review threshold'))
        print(box['name'], dict(zip(THREE_CLASSES, scores.tolist())))
    fig.tight_layout(); plt.show()
else: print('Optional manual photo test skipped; the ZIP has already been downloaded.')

## Recovery after disconnect
Run only this cell in a new session to download the latest completed Drive export. Checkpoints preserve weights if training stopped, but a checkpoint still needs held-out evaluation/export before deployment. Do not rebuild a fresh untrained head and assume it contains your old training.

In [ ]:
from pathlib import Path
if globals().get('EXPORT_DONE_THIS_SESSION', False):
    print('This run already downloaded its export.')
else:
    from google.colab import drive, files
    drive.mount('/content/drive')
    runs = Path('/content/drive/MyDrive/ParkVision_ThreeClass_EfficientNetB0/runs')
    exports = sorted(runs.glob('*/parkvision_three_class_export.zip'))
    if exports: files.download(str(exports[-1]))
    else:
        print('No completed export. Saved checkpoints:')
        for path in sorted(runs.glob('*/best_*.keras')): print(path)

# Dataset sources and reporting
- PKLot: https://web.inf.ufpr.br/vri/databases/parking-lot-database/ ; mirror: https://www.kaggle.com/datasets/ammarnassanalhajali/pklot-dataset . Retains dataset terms/citation.
- Open Images V7 class list and V5 validation boxes: https://storage.googleapis.com/openimages/web/download_v7.html . Selected source images retain their original licenses; OriginalURL/License/Author are carried into manifest.csv. Verify original image terms before redistribution.
- TraCon: https://github.com/ikatsamenis/Cone-Detection ; paper https://arxiv.org/abs/2205.11830 . Pinned commit 7ac4af371e826b5124a9b4defaa3327bc817817e. Cite the authors. No explicit dataset license file was found in that pinned repository; confirm permission before redistributing its images or commercial use. Raw images are not bundled in the model export.
- EfficientNet inputs: https://keras.io/api/applications/efficientnet/ ; fine-tuning: https://keras.io/guides/transfer_learning/ . RGB 0..255, preprocessing inside EfficientNet.

PKLot is regrouped by capture day. The Open Images validation SOURCE POOL is repartitioned into this experiment's own train/validation/test groups by image, not reported as its official benchmark. TraCon source photos are regrouped with perceptual near-duplicate grouping before splitting, ignoring its original split. This reduces leakage but cannot prove all related scenes were found. Keep your own capture sessions together.
Public object and annotation-free background crops are bootstrap proxies. An annotation-free patch is not verified parking ground; an object in an image does not prove it obstructs a parking space. Site-labeled fixed-box crops establish that distinction. Do not reuse binary COCO 'unoccupied' crops: those can contain people/objects that now need a different label.
Report the new run's full holdout scores, class precision/recall/support, per-source scores, and coverage of the review filter. Do not reuse the old S2 score, tune settings against test errors, or describe public performance as site accuracy. The three-class argmax is separate from the app's 0.80 review threshold. Confidence is not verified accuracy. Any site data included in previous model training must not be used as a new independent test set.